# Experiment 10 — Shamir's Secret Sharing Scheme
(k, n) threshold: any k of n shares reconstruct the secret.

## Finite Field & Polynomial

In [ ]:
import secrets, random
from itertools import combinations

PRIME = 2**127 - 1   # Mersenne prime

def _eval(coeffs, x, p=PRIME):
    r=0
    for c in reversed(coeffs): r=(r*x+c)%p
    return r

def _finv(a, p=PRIME): return pow(a, p-2, p)   # Fermat's little theorem

print(f"Prime field: 2^127 - 1 = {PRIME}")


## Split Secret → Shares

In [ ]:
def split(secret, k, n, p=PRIME):
    assert secret < p and 2<=k<=n
    coeffs=[secret]+[secrets.randbelow(p) for _ in range(k-1)]
    return [(i, _eval(coeffs,i,p)) for i in range(1,n+1)]


## Reconstruct Secret (Lagrange Interpolation)

In [ ]:
def reconstruct(shares, p=PRIME):
    s=0; xs=[sh[0] for sh in shares]; ys=[sh[1] for sh in shares]; k=len(shares)
    for i in range(k):
        num=ys[i]; den=1
        for j in range(k):
            if i==j: continue
            num=(num*(-xs[j]))%p; den=(den*(xs[i]-xs[j]))%p
        s=(s+num*_finv(den,p))%p
    return s

def txt2int(t): return int.from_bytes(t.encode('utf-8'),'big')
def int2txt(n): return n.to_bytes((n.bit_length()+7)//8,'big').decode()


## Demo — Integer Secret (3-of-5)

In [ ]:
secret=1234567890; k,n=3,5
shares=split(secret,k,n)
print(f"Secret   : {secret}")
print(f"Threshold: {k}-of-{n}")
print()
for x,y in shares: print(f"  Share {x}: {y}")

# Reconstruct with exactly k
r1=reconstruct(shares[:k])
print(f"\nUsing shares [1,2,3]: {r1}")
assert r1==secret; print("✓ Correct")

# Reconstruct with k+1
r2=reconstruct(shares[:k+1])
assert r2==secret; print(f"Using shares [1,2,3,4]: {r2}  ✓")

# Below threshold
r3=reconstruct(shares[:k-1])
print(f"Using only {k-1} shares : {r3}  (wrong — security holds)")
assert r3!=secret; print("✓ Below threshold: wrong answer")


## Demo — Text Secret (2-of-4)

In [ ]:
text="TopSecret!"; ti=txt2int(text); k2,n2=2,4
shares2=split(ti,k2,n2)
print(f"Secret : '{text}' → {ti}")
print(f"Shares (2-of-4):")
for x,y in shares2: print(f"  Share {x}: {y}")

chosen=random.sample(shares2,k2); chosen.sort(key=lambda s:s[0])
rec=reconstruct(chosen)
print(f"\nRecovered using shares {[s[0] for s in chosen]}: '{int2txt(rec)}'")
assert int2txt(rec)==text; print("✓ Text secret reconstructed correctly")


## Exhaustive Subset Verification

In [ ]:
ok=True
for combo in combinations(range(n2),k2):
    sub=[shares2[i] for i in combo]
    r=reconstruct(sub)
    match=r==ti
    print(f"  Shares {[s[0] for s in sub]}: {'✓' if match else '✗'}")
    if not match: ok=False
assert ok
print(f"\n✓ All C({n2},{k2})={len(list(combinations(range(n2),k2)))} subsets reconstruct correctly")
print()
print("Perfect information-theoretic security:")
print("  Any k-1 shares reveal ZERO information about the secret.")
